# preRA cohort scRNA analysis in Python for Monocytes 
- CertPro



In [ ]:
import h5py
import scipy.sparse as scs
import pandas as pd
import anndata
import os
import glob
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import median_abs_deviation
import scanpy as sc
# import sc_toolbox as sct
import decoupler as dc

In [ ]:
anndata.__version__

In [ ]:
# sc.settings.n_jobs = 58

In [ ]:
# define some color patterns for plotting
nejm_color = ["#BC3C29FF", "#0072B5FF", "#E18727FF", "#20854EFF", "#7876B1FF", "#6F99ADFF", "#FFDC91FF", "#EE4C97FF"]
jama_color = ["#374E55FF", "#DF8F44FF", "#00A1D5FF", "#B24745FF", "#79AF97FF", "#6A6599FF", "#80796BFF"]

In [ ]:
# define working path
data_path = '/home/jupyter/data/ra_longitudinal/reference_data/Alivernini_et_al_tissue_Macrophage/'
mtx_path = '/home/jupyter/data/ra_longitudinal/reference_data/Alivernini_et_al_tissue_Macrophage/expression_data'
fig_path = '/home/jupyter/data/ra_longitudinal/figures/mono/validation/'
meta_path = '/home/jupyter/github/ra-longitudinal/metadata/'
output_path = '/home/jupyter/data/ra_longitudinal/output_results/validation/'

# define a project name
proj_name = 'ALTRA_scRNA_mono_validation'
# sc.set_figure_params(fig_path)
sc.settings.figdir = fig_path
sc.settings.autosave=False
sc.set_figure_params(vector_friendly=True, dpi_save=300)

In [ ]:
# set fig size
plt.rcParams['figure.figsize'] = [10, 8]

# load data

# Analyze expression in paired comparison

In [ ]:
# load pair metadata 
pair_meta = pd.read_csv('/home/jupyter/data/ra_longitudinal/output_results/' + 'AIM3_paired_pre_post_conversion_samples.csv')
pair_meta.head()

In [ ]:
# load data
pair_mono_adata = sc.read_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

In [ ]:
pair_mono_adata

In [ ]:
pair_mono_adata.obs['AIFI_L3_new'].value_counts()

In [ ]:
pair_mono_adata.obs['celltype_status'] = pair_mono_adata.obs['AIFI_L3_new'].astype('str') + pair_mono_adata.obs['status'].astype('str')

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt.png',dendrogram=True)


In [ ]:
pair_mono_adata

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt_status.png',dendrogram=True)

In [ ]:
# plot the markers gene list from the paper
gene_list = ['NFKBIA', 'TNF', 'IL1B', "CCL3", 'CCL4', 'ICAM1', 'FOLR2', 
            'CD14', 'FCGR3A', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']

sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3",  standard_scale='var', swap_axes=True,
              save=  proj_name+'_AIFI_L3_target_genes.pdf')


In [ ]:
with plt.rc_context({"figure.figsize": (8, 4)}):
    sc.pl.violin(pair_mono_adata, ["TNF"], groupby="AIFI_L3", 
                 save=  proj_name+'_rna_TNF_violin.png',
                 inner="box", rotation=90)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(pair_mono_adata, svd_solver="arpack", use_highly_variable=True)
# plot the principle component variance explained
sc.pl.pca_variance_ratio(pair_mono_adata, log=True)

In [ ]:
pair_mono_adata

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['AIFI_L3_new', 'leiden_1'], legend_loc='on data',
    basis='X_tsne',
    frameon=False, ncols=2,
    save=  proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
# set order for status column
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('str')
pair_mono_adata.obs.loc[pair_mono_adata.obs['status']=='pre_conv', 'status'] = 'pre-disease'
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('category').cat.reorder_categories(
    ['pre-disease', 'conversion'], ordered=True)

In [ ]:
sc.tl.embedding_density(pair_mono_adata, basis='tsne', groupby='status')

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=14) 
sc.pl.embedding_density(pair_mono_adata, basis='tsne',
                        key='tsne_density_status', 
                        ncols=2, 
                        color_map= 'magma',
                       save=  proj_name+'_paired_tsne_density_status.pdf')

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['status', 'leiden_1', 'AIFI_L3_new'],#legend_loc='on data',
    basis='X_tsne',
    save=  proj_name+'_paired_status.png'
)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (400)}):
        ax = sc.pl.embedding(
        pair_mono_adata,
        color=['AIFI_L3_new'],  #legend_loc='on data', 
        legend_fontsize='small',
        frameon=False, basis='X_tsne',
        save = proj_name+'_paired_AIFI_L3_new.pdf'
    )


In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['leiden_1'], legend_loc='on data',
    frameon=False, basis='X_tsne',
    save = proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['TNF', 'IL1B', 'CCL3', 'NFKBIA'],#legend_loc='on data',
    basis='X_tsne',
        vmin='p1',
    vmax='p99',
    frameon=False, ncols=2,
    save=  proj_name+'_marker_genes.png'
)

In [ ]:
sc.pl.violin(pair_mono_adata, ['TNF'], groupby='AIFI_L3', rotation=90, save=proj_name+'TNF_violinplot.png')

In [ ]:
pair_mono_adata

In [ ]:
# save data
pair_mono_adata.write_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

### analyze the clusters

In [ ]:
# save data
pair_mono_adata = sc.read_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

### plot Fig 2I

In [ ]:
gene_list = ['IL1B', 'TNF', "CCL3", 'CCL4','CXCL3', 'CXCL8','CXCL10', 'ICAM1', 'FOLR2' ,'NFKBIA', 'NLRP3',
            'CD14', 'FCGR3A', 'HLA-DRA', 'CCR2', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']
cm = 1/2.54  # centimeters in inches
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3", standard_scale='var', figsize=[10.73/1.2, 3.21/1.2],
              save = proj_name+'_paired_samples_AIFI_L3.pdf', 
              dendrogram=True)

#ax.legend_.set_title("Cell type")
# Change Legend location
# ax.legend_.set_bbox_to_anchor((-0.2, -0.7))

In [ ]:
# Create the dotplot and return the figure
dotplot = sc.pl.dotplot(pair_mono_adata, gene_list, groupby="AIFI_L3", standard_scale='var', 
                        dendrogram=False, return_fig=True)
# Extract the figure and axes from the DotPlot object
axes = dotplot.get_axes()

In [ ]:
gene_list = ['NFKBIA', 'TNF', 'IL1B', "CCL3", 'CCL4', 'ICAM1', 'FOLR2', 
            'CD14', 'FCGR3A', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']
IL1B, TNF, CCL3, CCL4, CXCL3, CXCL8
sc.pl.dotplot(pair_mono_adata, gene_list, "leiden_1", standard_scale='var',
              save = proj_name+'_paired_samples_leiden_1_dotpolt.png', 
              dendrogram=True)


## compare IL1b monocyte with the core CD14 monocyte

In [ ]:
pair_mono_adata.obs['AIFI_L3'].unique()


In [ ]:
# test for top genes that seperate the cells
cluster_name = 'AIFI_L3'
sc.tl.rank_genes_groups(pair_mono_adata, groupby=cluster_name, 
                        groups= ['IL1B+ CD14 monocyte'], reference='Core CD14 monocyte',
                        method='wilcoxon', key_added='wilcoxon_il1b_vs_core_cd14')

In [ ]:
il1bvscore_cd14_degs = sc.get.rank_genes_groups_df(pair_mono_adata, key= 'wilcoxon_il1b_vs_core_cd14',
                                              pval_cutoff=0.05,  group=None)
il1bvscore_cd14_degs = il1bvscore_cd14_degs.reindex(il1bvscore_cd14_degs['pvals_adj'].sort_values(ascending=True).index)
il1bvscore_cd14_degs.to_csv(output_path + proj_name + cluster_name+'_il1bvscore_cd14_wilcoxon_sig_degs.csv')

In [ ]:
il1bvscore_cd14_degs = il1bvscore_cd14_degs.set_index('names')
il1bvscore_cd14_degs

In [ ]:
plot_volcano_df(
    il1bvscore_cd14_degs,
    x='logfoldchanges',
    y='pvals_adj',
    #genes=['IL1B', 'CCL3', 'CCL4', 'BTG2', 'EGR1', 'NFKB1A', 'TNF'], 
    lFCs_thr = 0.1, 
    sign_thr = 0.01,
    figsize=(5, 5), dpi=500,
    save= fig_path + proj_name+ cluster_name+ '_il1bvscore_cd14_degs_top40_valcano.pdf'
    )


In [ ]:
# test for top genes that seperate the cells
cluster_name = 'leiden_1'
sc.tl.rank_genes_groups(pair_mono_adata, groupby=cluster_name,
                        method='wilcoxon', key_added='leiden_1_wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(pair_mono_adata, n_genes=25, sharey=False, key="leiden_1_wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    pair_mono_adata, groupby=cluster_name, standard_scale="var", 
    n_genes=10, key=cluster_name +"_wilcoxon",
    save= '_'+ proj_name+ cluster_name + '_wilcoxon_top_genes_dotplot_scale.png'
)

In [ ]:
# output the deg list 
leiden_deg = sc.get.rank_genes_groups_df(pair_mono_adata, key= cluster_name + '_wilcoxon',
                                              pval_cutoff=None,  group=None).rename(
    {'group':cluster_name},  axis='columns')
leiden_deg['direction'] = np.where(leiden_deg['logfoldchanges']>0, 'up', 'down')
leiden_deg = leiden_deg.reindex(leiden_deg['scores'].abs().sort_values(ascending=False).index)
leiden_deg.to_csv(output_path + proj_name + cluster_name+'_wilcoxon_degs.csv')

# load data from Stefano Alivernini et al 
https://www.nature.com/articles/s41591-020-0939-8

In [ ]:
# load file_path
mtx_files = glob.glob(mtx_path + '**/*.mtx', recursive=True)
barcode_files = glob.glob(mtx_path + '**/*.barcodes.tsv', recursive=True)
gene_files = glob.glob(mtx_path + '**/*.genes.tsv', recursive=True)
len(mtx_files)

In [ ]:
# create data frames to capture file path
mtx_df = pd.DataFrame({'mtx_files':mtx_files})
mtx_df['sample_id'] = mtx_df['mtx_files'].str.extract(pat='(SA\\d\\d\\dP|SA\\d\\d\\d|HC\\d\\d\\d\\d)')
mtx_df = mtx_df.set_index('sample_id')
barcode_df = pd.DataFrame({'barcode_files':barcode_files})
barcode_df['sample_id'] = barcode_df['barcode_files'].str.extract(pat='(SA\\d\\d\\dP|SA\\d\\d\\d|HC\\d\\d\\d\\d)')
barcode_df = barcode_df.set_index('sample_id')
gene_df = pd.DataFrame({'gene_files':gene_files})
gene_df['sample_id'] = gene_df['gene_files'].str.extract(pat='(SA\\d\\d\\dP|SA\\d\\d\\d|HC\\d\\d\\d\\d)')
gene_df = gene_df.set_index('sample_id')

In [ ]:
mtx_df.index[mtx_df.index.duplicated()]

In [ ]:
files_df = pd.concat([mtx_df, barcode_df, gene_df], axis=1)
files_df.head()

In [ ]:
# load metadata 
mc_meta = pd.read_excel(data_path + 'E-MTAB-8322_metadata.xlsx')
mc_meta = mc_meta.rename(columns={'Source Name':'sample_id'})
mc_meta = mc_meta.drop_duplicates(['sample_id'])
mc_meta.head()

In [ ]:
mc_meta.shape

In [ ]:
# create a function to read data
def read_mtx(files_df, sample_id):
    mtx_path = files_df.loc[sample_id, 'mtx_files']
    barcode_path = files_df.loc[sample_id, 'barcode_files']
    gene_path = files_df.loc[sample_id, 'gene_files']
    # constrauct the anndata
    adata = sc.read_mtx(mtx_path)
    adata_bc = pd.read_csv(barcode_path, header=None).rename(columns={0:'index'})
    adata_features = pd.read_csv(gene_path,header=None)
    adata = adata.T
    adata.obs = adata_bc
    adata.obs['sample_id'] = sample_id
    adata.var = adata_features[0].str.split(pat="\t", n=2, expand=True).rename(columns={0:'ENSG_id', 1:'gene_name'})
    adata.var.index= adata.var['ENSG_id']
    return(adata)

In [ ]:
# read a test dataset
adata_149 = read_mtx(files_df, 'SA149P')
# adata_149P = read_mtx('SA149P')

In [ ]:
%%capture output
# read all the files in the folder
sample_ids = files_df.index.unique().tolist()
adatas = [read_mtx(files_df, sample_id) for sample_id in sample_ids]
# merge them into a anndata file toghether
#joint_adata = adatas[0].concatenate(adatas[1:])

In [ ]:
# run scrublet
import scrublet
for adata in adatas:
    print('scrubbing ' + adata.obs['sample_id'].unique())
    scrub = scrublet.Scrublet(adata.X)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose = False)
    adata.obs['doublet_scores'] = doublet_scores
    adata.obs['predicted_doublets'] = predicted_doublets

In [ ]:
adatas[0].var['gene_name'][adatas[0].var['gene_name'].duplicated()]

In [ ]:
adatas[0]

In [ ]:
# combined the dataset
joint_adata = sc.concat(adatas, join='inner')

In [ ]:
gene_names = adatas[0].var.reindex(joint_adata.var.index)
all(joint_adata.var.index == gene_names.index)

In [ ]:
joint_adata.var['gene_name'] = gene_names['gene_name']

In [ ]:
joint_adata.obs.index = joint_adata.obs['sample_id'] + '_' + joint_adata.obs['index']

In [ ]:
joint_adata.obs.index.duplicated().any()

In [ ]:
len(joint_adata.obs['sample_id'].unique())

In [ ]:
# add metadata
joint_adata.obs = joint_adata.obs.merge(mc_meta, how='left', on=['sample_id'])

In [ ]:
joint_adata

In [ ]:
# save the data
joint_adata.write_h5ad(data_path + 'Validation_RA_scRNA_tissue_macrophages.h5ad')

## basic qc 

In [ ]:
sc.pp.filter_cells(joint_adata, min_genes=200)

In [ ]:
sc.pp.filter_genes(joint_adata, min_cells=3)

In [ ]:
joint_adata

In [ ]:
joint_adata.var

In [ ]:
# mitochondrial genes
joint_adata.var["mt"] = joint_adata.var['gene_name'].str.startswith("MT-")
# ribosomal genes
joint_adata.var["ribo"] = joint_adata.var['gene_name'].str.startswith(("RPS", "RPL"))
# hemoglobin genes
joint_adata.var["hb"] = joint_adata.var['gene_name'].str.contains(("^HB[^(P)]"))

In [ ]:
sc.pp.calculate_qc_metrics(
    joint_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)


In [ ]:
joint_adata

In [ ]:
sc.pl.violin(joint_adata, ['pct_counts_hb',  'pct_counts_ribo', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.violin(joint_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(joint_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(joint_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
p1 = sns.displot(joint_adata.obs["total_counts"], bins=100, kde=False)
p3 = sc.pl.scatter(joint_adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
joint_adata.obs["outlier"] = (
    is_outlier(joint_adata, "log1p_total_counts", 5)
    | is_outlier(joint_adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(joint_adata, "pct_counts_in_top_20_genes", 5)
)
joint_adata.obs.outlier.value_counts()

In [ ]:
# filter cell based on 
joint_adata.obs["mt_outlier"] = is_outlier(joint_adata, "pct_counts_mt", 5) | (
    joint_adata.obs["pct_counts_mt"] > 10
)
joint_adata.obs.mt_outlier.value_counts()

In [ ]:
joint_adata.obs['predicted_doublets'].value_counts()

In [ ]:
print(f"Total number of cells: {joint_adata.n_obs}")
joint_adata_fl = joint_adata[(~joint_adata.obs.outlier) & (~joint_adata.obs.mt_outlier) &
     (joint_adata.obs.predicted_doublets==False)].copy()

print(f"Number of cells after filtering of low quality cells: {joint_adata_fl.n_obs}")

In [ ]:
59007/64510

In [ ]:
# remove two donors SA139, SA225 per paper
joint_adata_fl = joint_adata_fl[~joint_adata_fl.obs['sample_id'].isin(['SA139', 'SA225'])].copy()
joint_adata_fl

In [ ]:
# save the data
joint_adata_fl.write_h5ad(data_path + 'Validation_RA_scRNA_tissue_macrophages_filtered.h5ad')

## Normalize RNA

In [ ]:
# load data
joint_adata_fl = sc.read_h5ad(data_path +'Validation_RA_scRNA_tissue_macrophages_filtered.h5ad')

In [ ]:
joint_adata_fl

In [ ]:
joint_adata_fl.layers['counts'] = joint_adata_fl.X.copy()

In [ ]:
# # some basic filtering
# # %%time
# sc.pp.filter_genes(joint_adata, min_counts=3)
# sc.pp.filter_cells(joint_adata, min_counts=3)

In [ ]:
p1 = sns.histplot(joint_adata_fl.obs["total_counts"], bins=100, kde=False)

In [ ]:
# joint_adata_fl.is_view

In [ ]:
# log1p transform
scales_counts = sc.pp.normalize_total(joint_adata_fl, target_sum=None, inplace=False)
joint_adata_fl.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1 = sns.histplot(joint_adata_fl.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts")
p2 = sns.histplot(joint_adata_fl.layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("Shifted logarithm")
plt.show()

In [ ]:
joint_adata_fl.X = joint_adata_fl.layers['counts'].copy()

In [ ]:
%%time
# cpm normalization
sc.pp.normalize_total(joint_adata_fl, target_sum=1e4, inplace=True)
sc.pp.log1p(joint_adata_fl)


In [ ]:
joint_adata_fl.X = joint_adata_fl.layers['log1p_norm'].copy()

In [ ]:
# # Find highly variable genes
# sc.pp.highly_variable_genes(
#     joint_adata_fl,
#     n_top_genes=3000,
#     layer="counts",
#     flavor="seurat_v3",
#    # batch_key="batch"
# )

In [ ]:
# %%time
sc.pp.highly_variable_genes(joint_adata_fl, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(joint_adata_fl)

In [ ]:
joint_adata_fl.var.highly_variable.value_counts()

In [ ]:
joint_adata_fl.raw = joint_adata_fl

In [ ]:
joint_adata_fl.raw.X

In [ ]:
# joint_adata_fl.X = joint_adata_fl.layers['log1p_norm'].copy()

In [ ]:
# joint_adata_fl.X[1:30, 1:30].toarray() == joint_adata_fl.layers['log1p_norm'][1:30, 1:30].toarray()

In [ ]:
sc.pp.scale(joint_adata_fl, max_value=10)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(joint_adata_fl, svd_solver="arpack", use_highly_variable=True)

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(joint_adata_fl, log=True)

In [ ]:
sc.pl.pca_scatter(joint_adata_fl, color=["pct_counts_ribo", 'pct_counts_mt'])

In [ ]:
joint_adata_fl

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(joint_adata_fl, log=True)

In [ ]:
sc.pp.neighbors(joint_adata_fl, n_neighbors=10, n_pcs=11)
sc.tl.umap(joint_adata_fl)

In [ ]:
sc.pl.pca_loadings(joint_adata_fl, components = '1,2,3')

In [ ]:
sc.tl.leiden(joint_adata_fl, key_added="leiden_0_5", resolution=0.5)

In [ ]:
sc.pl.umap(
    joint_adata_fl,
    color=['sample_id', 'leiden_0_5', 'disease',  'response to treatment', 'sex'],
    ncols=3,
    frameon=False,wspace=0.5,
    save=  proj_name+'_rna_umap.png'
)

In [ ]:
joint_adata_fl.obsm['X_orginal_umap'] = joint_adata_fl.obsm['X_umap'].copy()

In [ ]:
# run harmony
import scanpy.external as sce
sce.pp.harmony_integrate(joint_adata_fl, 'individual',  max_iter_harmony = 50, adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(joint_adata_fl, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(joint_adata_fl)

In [ ]:
sc.tl.leiden(joint_adata_fl, key_added="leiden_0_2", resolution=0.2)

In [ ]:
sc.pl.umap(
    joint_adata_fl,
    color=['sample_id', 'leiden_0_2', 'disease',  'response to treatment'],
    ncols=3,
    frameon=False,wspace=0.5,
    save=  proj_name+'_rna_harmony_umap.png'
)


In [ ]:
joint_adata_fl.var['gene_name'][joint_adata_fl.var['gene_name'].duplicated()]

In [ ]:
# joint_adata_fl.var.index = joint_adata_fl.var['gene_name'].astype('str')
# joint_adata_fl.var_names_make_unique()
# joint_adata_fl.var.index = joint_adata_fl.var.index.astype('category')

In [ ]:
joint_adata_fl.X.shape

In [ ]:
joint_adata_fl.var

In [ ]:
sc.pl.umap(
    joint_adata_fl,
    color=[ 'leiden_0_2'],
    frameon=False,
    save=  proj_name+'_rna_harmony_umap_leiden_0_2.png'
)

In [ ]:
sc.pl.umap(
    joint_adata_fl, gene_symbols='gene_name',
    color=['TNF', 'CCL3', 'CCL4', 'ICAM1', 'IL1B'],
    save=  proj_name+'_rna_harmony_umap_gene_exprs.png'
)

In [ ]:
joint_adata_fl

In [ ]:
joint_adata_fl.var.loc[joint_adata_fl.var['gene_name'].isin(['CD3', 'CD8', 'CD19',  'CD14', 'CD16', 'HLA-DRA'])]

In [ ]:
sc.pl.umap(
    joint_adata_fl, gene_symbols='gene_name',
    color=['CD19',  'CD14', 'HLA-DRA'],
    save=  proj_name+'_rna_harmony_umap_gene_exprs.png'
)

In [ ]:
joint_adata_fl

In [ ]:
sc.pl.embedding(
    joint_adata_fl, basis='X_orginal_umap', gene_symbols='gene_name',
    color=['TNF', 'CCL3', 'CCL4', 'ICAM1', 'IL1B'],
    save=  proj_name+'_rna_umap_gene_exprs.png'
)

In [ ]:
# save the data
joint_adata_fl.write_h5ad(data_path + 'Validation_RA_scRNA_tissue_macrophages_filtered.h5ad')

## run another umap just based on the differntial genes from the paper

In [ ]:
# load the degs table
cluster_degs = pd.read_excel(data_path + '/41591_2020_939_MOESM3_ESM_cluster_degs.xlsx')

In [ ]:
cluster_degs

In [ ]:
joint_adata_fl.var.loc[joint_adata_fl.var['gene_name'].isin(cluster_degs['gene symbol'])]

In [ ]:
joint_adata_sub = joint_adata_fl[:, joint_adata_fl.var['gene_name'].isin(cluster_degs['gene symbol'])].copy()
joint_adata_sub

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(joint_adata_sub, svd_solver="arpack", )

In [ ]:
# run harmony
import scanpy.external as sce
sce.pp.harmony_integrate(joint_adata_sub, 'individual',  max_iter_harmony = 50, adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(joint_adata_sub, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(joint_adata_sub)

In [ ]:
sc.tl.leiden(joint_adata_sub, key_added="leiden_0_2", resolution=0.2)

In [ ]:
sc.pl.umap(
    joint_adata_sub,
    color=['sample_id', 'leiden_0_2', 'disease',  'response to treatment'],
    ncols=3,
    frameon=False,wspace=0.5,
    save=  proj_name+'_rna_harmony_cluster_degs_umap.png'
)


In [ ]:
sc.pl.umap(
    joint_adata_sub, gene_symbols='gene_name',
    color=['TNF', 'CCL3', 'CCL4', 'ICAM1', 'IL1B'],
    save=  proj_name+'_rna_harmony_cluster_degs_umap_gene_exprs.png'
)

In [ ]:
cluster_degs_c8 = cluster_degs.loc[cluster_degs['cluster'] == 8]
cluster_degs_c8

In [ ]:
sc.pl.umap(
    joint_adata_sub, gene_symbols='gene_name',
    color=cluster_degs_c8['gene symbol'][0:9],
    save=  proj_name+'_rna_harmony_cluster_degs_umap_marker_gene_exprs.png'
)

# load merged dataset

In [ ]:
merge_matrix = sc.read_csv(mtx_path + '/Merged.matrix.csv', dtype='int')

In [ ]:
merge_anndata = merge_matrix.T

In [ ]:
merge_anndata.obs['barcodes'] = merge_anndata.obs.index.astype('str')
merge_anndata.obs['sample_id'] = merge_anndata.obs['barcodes'].str.extract(pat='(SA\\d\\d\\dP|SA\\d\\d\\d|HC\\d\\d\\d\\d)')

In [ ]:
# add metadata
merge_anndata.obs = merge_anndata.obs.merge(mc_meta, how='left', on=['sample_id'])

In [ ]:
# load the data
merge_anndata = sc.read_h5ad(data_path + 'Validation_RA_scRNA_tissue_macrophages_merged.h5ad')

In [ ]:
# mitochondrial genes
merge_anndata.var["mt"] = merge_anndata.var.index.str.startswith("MT-")
# ribosomal genes
merge_anndata.var["ribo"] = merge_anndata.var.index.str.startswith(("RPS", "RPL"))
# hemoglobin genes
merge_anndata.var["hb"] = merge_anndata.var.index.str.contains(("^HB[^(P)]"))

In [ ]:
merge_anndata.var.loc[merge_anndata.var.index.str.startswith("MT-")]

In [ ]:
sc.pp.calculate_qc_metrics(
    merge_anndata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)


In [ ]:
merge_anndata

In [ ]:
sc.pl.violin(merge_anndata, ['pct_counts_hb',  'pct_counts_ribo', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
merge_anndata.layers['counts'] = merge_anndata.X.copy()

In [ ]:
%%time
# cpm normalization
sc.pp.normalize_total(merge_anndata, target_sum=1e4, inplace=True)
sc.pp.log1p(merge_anndata)


In [ ]:
# %%time
sc.pp.highly_variable_genes(merge_anndata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(merge_anndata)

In [ ]:
merge_anndata.var.highly_variable.value_counts()

In [ ]:
merge_anndata.raw = merge_anndata

In [ ]:
merge_anndata.raw.X

In [ ]:
# joint_adata_fl.X = joint_adata_fl.layers['log1p_norm'].copy()

In [ ]:
# joint_adata_fl.X[1:30, 1:30].toarray() == joint_adata_fl.layers['log1p_norm'][1:30, 1:30].toarray()

In [ ]:
sc.pp.scale(merge_anndata, max_value=10)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(merge_anndata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
sc.pl.pca_scatter(merge_anndata, color=["pct_counts_ribo", 'pct_counts_mt'])

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(merge_anndata, log=True)

In [ ]:
sc.pp.neighbors(merge_anndata, n_neighbors=10, n_pcs=8)
sc.tl.umap(merge_anndata)

In [ ]:
sc.pl.pca_loadings(merge_anndata, components = '1,2,3')

In [ ]:
sc.tl.leiden(merge_anndata, key_added="leiden_0_5", resolution=0.5)

In [ ]:
sc.pl.umap(
    merge_anndata,
    color=['sample_id', 'leiden_0_5', 'disease',  'response to treatment', 'sex'],
    ncols=3,
    frameon=False,wspace=0.5,
    save=  proj_name+'_rna_umap.png'
)

In [ ]:
merge_anndata.obsm['X_orginal_umap'] = merge_anndata.obsm['X_umap'].copy()

In [ ]:
# run harmony
import scanpy.external as sce
sce.pp.harmony_integrate(merge_anndata, 'individual',  max_iter_harmony = 50, adjusted_basis='X_pca_harmony')

In [ ]:
sc.pp.neighbors(merge_anndata, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(merge_anndata)

In [ ]:
sc.tl.leiden(merge_anndata, key_added="leiden_0_6", resolution=0.6)

In [ ]:
sc.pl.umap(
    merge_anndata,
    color=['sample_id', 'leiden_0_2','leiden_0_6', 'disease',  'response to treatment'],
    ncols=3,
    frameon=False,
    save=  proj_name+'_rna_harmony_umap.png'
)


In [ ]:
sc.pl.umap(
    merge_anndata,
    color=['TNF', 'CCL3', 'CCL4', 'ICAM1', 'IL1B'],
    save=  proj_name+'_rna_harmony_cluster_degs_umap_marker_gene_exprs.png'
)

In [ ]:
# save the data
merge_anndata.write_h5ad(data_path + 'Validation_RA_scRNA_tissue_macrophages_merged.h5ad')

In [ ]:
### label transfer from our datas

# Session Info

In [ ]:
import sinfo
sinfo.sinfo(write_req_file = False)